# Étape 2 : entraînement des modèles de détection de toxicité

Données : `data/clean/train_clean.csv` (colonne `comment_text` + 6 labels). Multi-label : 6 sorties sigmoïde, perte BCE avec `pos_weight`.
Trois modèles comparés **sur les mêmes données** (consigne du prof) : BiLSTM, DistilBERT, LAYA.

- **Entraînement commun** : les mêmes `N_COMMUN` lignes (stratifiées) tirées des 80 % de `train_clean.csv`. LAYA n'est pas entraîné (zéro-shot : voir sa section).
- **Validation** : les 20 % restants (graine 42). Elle sert à **choisir le seuil de décision de chaque label** (celui qui maximise le F1). BiLSTM et DistilBERT l'utilisent en entier ; LAYA, trop lent, en utilise `N_VAL_LAYA` lignes.
- **Test officiel commun** : les mêmes `N_TEST_COMMUN` lignes de `test_clean.csv` (avec `test_labels.csv`, **sans les lignes à `-1`**, non évaluées par Kaggle). Il n'est regardé qu'à la fin, sans aucun réglage dessus.

Les choix et le tableau comparatif final sont consignés dans `CLAUDE.md`.

In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"   # doit être défini AVANT d'importer torch (repli CPU pour les opérations non gérées par MPS)

import time, json, random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, f1_score
from tqdm.auto import tqdm

# --- Chemins (relatifs à la racine du projet) ---
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # racine du projet (notebook lancé depuis notebooks/ ou depuis la racine)
CHEMIN_CSV = RACINE / "data" / "clean" / "train_clean.csv"
CHEMIN_TEST = RACINE / "data" / "clean" / "test_clean.csv"         # test officiel (textes nettoyés)
CHEMIN_TEST_LABELS = RACINE / "data" / "test_labels.csv"           # labels du test officiel ; -1 = ligne non évaluée
DOSSIER_MODELES = RACINE / "models"          # non commité (voir .gitignore)
CHEMIN_CLAUDE_MD = RACINE / "CLAUDE.md"        # le tableau comparatif y est écrit à la fin
LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# --- Paramètres communs ---
SEED = 42                # même graine et même split pour tous les modèles
SEUIL = 0.5              # seuil de décision pour précision / rappel / F1

# --- Paramètres BiLSTM ---
VOCAB_LSTM, LONGUEUR_LSTM, BATCH_LSTM = 30_000, 200, 128
EMB_LSTM, HIDDEN_LSTM, EPOCHS_LSTM, LR_LSTM = 128, 64, 3, 1e-3

# --- Jeux de données COMMUNS aux trois modèles (consigne du prof) ---
N_COMMUN = 30_000         # lignes d'entraînement : les MÊMES pour BiLSTM, DistilBERT et (zéro-shot) LAYA
N_TEST_COMMUN = 10_000    # lignes du test officiel (sans les -1) : les MÊMES pour les trois modèles
N_VAL_LAYA = 3000         # lignes de validation pour LAYA (inférence lente) ; BiLSTM et DistilBERT utilisent toute la validation
REENTRAINER_LSTM = False  # False : recharge models/bilstm s'il existe (évite ~3 min) ; True : réentraîne
REENTRAINER_BERT = False  # False : recharge models/distilbert s'il existe (évite ~40 min) ; True : réentraîne
TEMPS_TRAIN_BERT_CONNU = None   # si DistilBERT est rechargé sans temps enregistré : temps d'entraînement mesuré avant (en s)

# --- Paramètres DistilBERT ---
NOM_BERT = "distilbert-base-uncased"
LONGUEUR_BERT, BATCH_BERT, EPOCHS_BERT, LR_BERT = 128, 16, 2, 2e-5

# Tout en float32 ; périphérique MPS (Mac M3) si disponible, sinon CPU
torch.set_default_dtype(torch.float32)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Périphérique :", DEVICE)


def fixer_graine(seed: int = SEED):
    """Rend l'entraînement reproductible."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def synchroniser():
    """Attend la fin des calculs GPU pour que les chronométrages soient justes."""
    if DEVICE.type == "mps":
        torch.mps.synchronize()


fixer_graine()
DOSSIER_MODELES.mkdir(exist_ok=True)
DOSSIER_PRED = DOSSIER_MODELES / "predictions"      # probabilités et temps de chaque modèle (voir evaluer_jeu)
DOSSIER_PRED.mkdir(exist_ok=True)

Périphérique : mps


/Users/situn/Situu/Projects/depl_ai_cyber/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Données : split 80/20 et jeu d'entraînement commun

In [2]:
df = pd.read_csv(CHEMIN_CSV)
df["comment_text"] = df["comment_text"].fillna("")
textes = df["comment_text"].to_numpy()
Y = df[LABELS].to_numpy(dtype=np.float32)

# Split fixe : les mêmes indices servent à tous les modèles
indices = np.arange(len(df))
idx_train, idx_val = train_test_split(indices, test_size=0.2, random_state=SEED)
print(f"{len(df)} lignes -> {len(idx_train)} entraînement (80 %) / {len(idx_val)} validation (20 %)")
print("Part de positifs par label (train) :")
print(pd.Series(Y[idx_train].mean(axis=0), index=LABELS).round(4).to_string())

# Jeu d'entraînement COMMUN aux trois modèles : N_COMMUN lignes tirées des 80 %, stratifiées.
# Stratification multi-label : sur la combinaison exacte des 6 labels ; les combinaisons rares (< 10) sont regroupées.
cle = pd.Series(["".join(map(str, ligne.astype(int))) for ligne in Y[idx_train]])
cle[cle.map(cle.value_counts()) < 10] = "rare"
if N_COMMUN < len(idx_train):
    idx_commun, _ = train_test_split(idx_train, train_size=N_COMMUN, stratify=cle.to_numpy(), random_state=SEED)
else:
    idx_commun = idx_train
print(f"Entraînement commun : {len(idx_commun)} lignes. Part de positifs :")
print(pd.Series(Y[idx_commun].mean(axis=0), index=LABELS).round(4).to_string())


def calculer_pos_weight(y: np.ndarray) -> torch.Tensor:
    """pos_weight = négatifs / positifs, par label : compense le déséquilibre des classes."""
    positifs = y.sum(axis=0)
    return torch.tensor((len(y) - positifs) / np.maximum(positifs, 1), dtype=torch.float32, device=DEVICE)

159571 lignes -> 127656 entraînement (80 %) / 31915 validation (20 %)
Part de positifs par label (train) :
toxic            0.0959
severe_toxic     0.0100
obscene          0.0528
threat           0.0032
insult           0.0491
identity_hate    0.0087
Entraînement commun : 30000 lignes. Part de positifs :
toxic            0.0959
severe_toxic     0.0100
obscene          0.0527
threat           0.0032
insult           0.0490
identity_hate    0.0087


# Test officiel (Kaggle) : retrait des lignes à `-1`

In [3]:
# Test final : jamais utilisé pendant l'entraînement ni pour régler quoi que ce soit
df_off = pd.read_csv(CHEMIN_TEST)
labels_off = pd.read_csv(CHEMIN_TEST_LABELS)
assert (df_off["id"] == labels_off["id"]).all(), "test.csv et test_labels.csv ne sont pas alignés"

# Dans test_labels.csv, -1 = commentaire NON évalué par Kaggle : on le retire AVANT toute prédiction
non_evalue = (labels_off[LABELS] == -1).all(axis=1)
assert ((labels_off[LABELS] == -1).any(axis=1) == non_evalue).all(), "ligne partiellement à -1 : inattendu"
garde = ~non_evalue.to_numpy()

textes_off = df_off["comment_text"].fillna("").to_numpy()[garde]
Y_off = labels_off[LABELS].to_numpy(dtype=np.float32)[garde]
print(f"Test officiel : {len(df_off)} lignes -> {len(textes_off)} évaluées ({int(non_evalue.sum())} lignes à -1 retirées)")

# Test COMMUN aux trois modèles : mêmes N_TEST_COMMUN lignes tirées au hasard (graine fixe)
if N_TEST_COMMUN is not None and N_TEST_COMMUN < len(textes_off):
    choix = np.sort(np.random.RandomState(SEED).choice(len(textes_off), N_TEST_COMMUN, replace=False))
    textes_off, Y_off = textes_off[choix], Y_off[choix]
print(f"Test commun retenu : {len(textes_off)} lignes")
print("Part de positifs par label (test officiel) :")
print(pd.Series(Y_off.mean(axis=0), index=LABELS).round(4).to_string())

Test officiel : 153164 lignes -> 63978 évaluées (89186 lignes à -1 retirées)
Test commun retenu : 10000 lignes
Part de positifs par label (test officiel) :
toxic            0.0981
severe_toxic     0.0058
obscene          0.0603
threat           0.0034
insult           0.0580
identity_hate    0.0125


# Évaluation commune et choix du seuil

In [4]:
RESULTATS = {}   # nom du modèle -> dict de résultats (alimenté par chaque modèle)


def choisir_seuils(y_vrai, proba):
    """Seuil de chaque label = celui qui maximise le F1 sur la VALIDATION (jamais sur le test).

    Un label sans aucun positif en validation garde le seuil par défaut SEUIL.
    """
    grille = np.arange(0.02, 0.99, 0.01)
    seuils = []
    for j in range(len(LABELS)):
        if y_vrai[:, j].sum() == 0:
            seuils.append(SEUIL)
            continue
        f1s = [f1_score(y_vrai[:, j], (proba[:, j] >= s).astype(int), zero_division=0) for s in grille]
        seuils.append(float(grille[int(np.argmax(f1s))]))
    return np.array(seuils)


def evaluer(nom, y_vrai, proba, temps_train, temps_inference, jeu="validation", seuils=None):
    """Précision, rappel, F1 (aux seuils donnés, sinon SEUIL) et ROC-AUC par label, + temps."""
    seuils = np.full(len(LABELS), SEUIL) if seuils is None else np.asarray(seuils)
    pred = (proba >= seuils).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_vrai, pred, average=None, zero_division=0)
    auc = []
    for j in range(len(LABELS)):
        try:
            auc.append(roc_auc_score(y_vrai[:, j], proba[:, j]))
        except ValueError:          # label absent du jeu : AUC non définie
            auc.append(float("nan"))
    par_label = pd.DataFrame({"seuil": seuils, "precision": p, "rappel": r, "f1": f, "roc_auc": auc}, index=LABELS)
    res = RESULTATS.setdefault(nom, {"temps_train_s": 0.0})
    if temps_train is not None:                 # le temps d'entraînement n'est donné qu'une fois
        res["temps_train_s"] = temps_train
    res[jeu] = {"par_label": par_label, "temps_inference_s": temps_inference,
                "ms_par_commentaire": 1000 * temps_inference / len(y_vrai), "n_lignes": len(y_vrai)}
    print(f"\n== {nom} [{jeu}] ==  inférence {temps_inference:.1f} s "
          f"({res[jeu]['ms_par_commentaire']:.2f} ms/commentaire) sur {len(y_vrai)} lignes")
    display(par_label.round(4))


def sauver_predictions(nom, jeu, proba, temps_train, temps_inference):
    """Enregistre les probabilités et les temps dans models/predictions/ : on peut refaire seuils et tableaux sans relancer le modèle."""
    cle = "".join(c if c.isalnum() else "_" for c in nom.lower()).strip("_")
    np.save(DOSSIER_PRED / f"{cle}_{jeu}.npy", proba)
    infos = {"temps_inference_s": temps_inference, "lignes": int(len(proba))}
    if temps_train is not None:
        infos["temps_train_s"] = temps_train
    (DOSSIER_PRED / f"{cle}_{jeu}.json").write_text(json.dumps(infos), encoding="utf-8")


def evaluer_jeu(nom, jeu, y_vrai, proba, temps_train, temps_inference):
    """Évalue au seuil fixe 0,5 ET au seuil optimisé. Sur la validation, les seuils sont choisis ; sur le test, ils sont appliqués."""
    sauver_predictions(nom, jeu, proba, temps_train, temps_inference)      # enregistré avant tout calcul
    res = RESULTATS.setdefault(nom, {"temps_train_s": 0.0})
    if jeu == "validation":
        res["seuils"] = choisir_seuils(y_vrai, proba)
    evaluer(nom, y_vrai, proba, temps_train, temps_inference, jeu=jeu)                                # seuil 0,5
    evaluer(nom, y_vrai, proba, None, temps_inference, jeu=jeu + "_opt", seuils=res["seuils"])        # seuil optimisé (choisi sur la validation)

# Modèle 1 : BiLSTM (jeu d'entraînement commun)

In [5]:
# Vocabulaire construit sur le jeu d'entraînement COMMUN uniquement (les textes sont déjà nettoyés : minuscules, sans ponctuation)
compteur = Counter(mot for t in textes[idx_commun] for mot in t.split())
vocab = {"<pad>": 0, "<unk>": 1}
for mot, _ in compteur.most_common(VOCAB_LSTM - 2):
    vocab[mot] = len(vocab)


def encoder(liste_textes):
    """Texte -> suite d'identifiants, tronquée / complétée par des 0 à LONGUEUR_LSTM."""
    X = np.zeros((len(liste_textes), LONGUEUR_LSTM), dtype=np.int64)
    for i, t in enumerate(liste_textes):
        ids = [vocab.get(m, 1) for m in t.split()[:LONGUEUR_LSTM]] or [1]   # texte vide -> un jeton <unk>
        X[i, :len(ids)] = ids
    return torch.from_numpy(X)


X_train_l, X_val_l = encoder(textes[idx_commun]), encoder(textes[idx_val])
Y_train_l = torch.from_numpy(Y[idx_commun])
print("Vocabulaire :", len(vocab), "| X_train :", tuple(X_train_l.shape))

Vocabulaire : 30000 | X_train : (30000, 200)


In [6]:
class BiLSTM(nn.Module):
    """Embedding 128 -> LSTM bidirectionnel (64 par sens) -> max-pooling -> 6 logits."""

    def __init__(self, taille_vocab, emb=EMB_LSTM, hidden=HIDDEN_LSTM, n_labels=len(LABELS)):
        super().__init__()
        self.embedding = nn.Embedding(taille_vocab, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.sortie = nn.Linear(2 * hidden, n_labels)

    def forward(self, x):
        masque = (x != 0).unsqueeze(-1)                 # ignore le padding dans le pooling
        h, _ = self.lstm(self.embedding(x))
        h = h.masked_fill(~masque, -1e9)
        return self.sortie(self.dropout(h.max(dim=1).values))   # logits (la sigmoïde est dans la perte / à l'inférence)


@torch.no_grad()
def predire_lstm(modele, X, batch=512):
    modele.eval()
    sorties = [torch.sigmoid(modele(X[i:i + batch].to(DEVICE))).cpu()
               for i in tqdm(range(0, len(X), batch), desc="BiLSTM inférence")]
    return torch.cat(sorties).numpy()

In [7]:
chemin_lstm = DOSSIER_MODELES / "bilstm"
fichier_temps_lstm = chemin_lstm / "temps_train.json"
if (chemin_lstm / "bilstm.pt").exists() and not REENTRAINER_LSTM:
    # Modèle déjà entraîné sur le même jeu commun (même graine) : on le recharge au lieu de le réentraîner
    vocab_sauve = json.loads((chemin_lstm / "vocab.json").read_text(encoding="utf-8"))
    if vocab_sauve != vocab:                     # le modèle doit être utilisé avec le vocabulaire sur lequel il a appris
        vocab = vocab_sauve
        X_val_l = encoder(textes[idx_val])
        print("Vocabulaire du modèle sauvegardé utilisé (différent de celui recalculé)")
    modele_lstm = BiLSTM(len(vocab)).to(DEVICE)
    modele_lstm.load_state_dict(torch.load(chemin_lstm / "bilstm.pt", map_location=DEVICE))
    temps_train_lstm = json.loads(fichier_temps_lstm.read_text())["temps_train_s"] if fichier_temps_lstm.exists() else float("nan")
    print("BiLSTM rechargé depuis", chemin_lstm, "| temps d'entraînement :", temps_train_lstm)
else:
    fixer_graine()
    modele_lstm = BiLSTM(len(vocab)).to(DEVICE)
    perte = nn.BCEWithLogitsLoss(pos_weight=calculer_pos_weight(Y[idx_commun]))   # BCE + pos_weight
    optimiseur = torch.optim.Adam(modele_lstm.parameters(), lr=LR_LSTM)
    loader = DataLoader(TensorDataset(X_train_l, Y_train_l), batch_size=BATCH_LSTM, shuffle=True)

    synchroniser(); t0 = time.perf_counter()
    for epoque in range(EPOCHS_LSTM):
        modele_lstm.train()
        total = 0.0
        barre = tqdm(loader, desc=f"BiLSTM époque {epoque + 1}/{EPOCHS_LSTM}")
        for etape, (xb, yb) in enumerate(barre, 1):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimiseur.zero_grad()
            l = perte(modele_lstm(xb), yb)
            l.backward()
            optimiseur.step()
            total += l.item()
            barre.set_postfix(perte=f"{total / etape:.4f}")   # perte moyenne en direct
        print(f"  perte moyenne : {total / len(loader):.4f}")
    synchroniser(); temps_train_lstm = time.perf_counter() - t0

    # Sauvegarde : poids + vocabulaire (nécessaire pour réutiliser le modèle) + temps d'entraînement
    (DOSSIER_MODELES / "bilstm").mkdir(exist_ok=True)
    (DOSSIER_MODELES / "bilstm" / "temps_train.json").write_text(json.dumps({"temps_train_s": temps_train_lstm}))
    torch.save(modele_lstm.state_dict(), DOSSIER_MODELES / "bilstm" / "bilstm.pt")
    (DOSSIER_MODELES / "bilstm" / "vocab.json").write_text(json.dumps(vocab), encoding="utf-8")

# Inférence chronométrée sur toute la validation
synchroniser(); t0 = time.perf_counter()
proba_lstm = predire_lstm(modele_lstm, X_val_l)
synchroniser(); temps_inf_lstm = time.perf_counter() - t0

evaluer_jeu("BiLSTM", "validation", Y[idx_val], proba_lstm, temps_train_lstm, temps_inf_lstm)


BiLSTM rechargé depuis /Users/situn/Situu/Projects/depl_ai_cyber/models/bilstm | temps d'entraînement : 157.37056379200658


BiLSTM inférence: 100%|██████████| 63/63 [00:04<00:00, 15.16it/s]



== BiLSTM [validation] ==  inférence 4.2 s (0.13 ms/commentaire) sur 31915 lignes


,seuil,precision,rappel,f1,roc_auc
toxic,0.5,0.3547,0.8658,0.5032,0.9297
severe_toxic,0.5,0.1207,0.9221,0.2135,0.9764
obscene,0.5,0.3135,0.9055,0.4658,0.9576
threat,0.5,0.0182,0.8784,0.0357,0.9470
insult,0.5,0.2649,0.8984,0.4092,0.9502
identity_hate,0.5,0.0654,0.8163,0.1210,0.9260



== BiLSTM [validation_opt] ==  inférence 4.2 s (0.13 ms/commentaire) sur 31915 lignes


,seuil,precision,rappel,f1,roc_auc
toxic,0.84,0.6702,0.6378,0.6536,0.9297
severe_toxic,0.96,0.3940,0.4922,0.4377,0.9764
obscene,0.90,0.6259,0.6741,0.6491,0.9576
threat,0.96,0.0672,0.2162,0.1026,0.9470
insult,0.90,0.5572,0.6400,0.5957,0.9502
identity_hate,0.86,0.1362,0.5680,0.2197,0.9260


In [8]:
# Test officiel commun : BiLSTM (le modèle n'est plus modifié ici ; les seuils viennent de la validation)
X_off_l = encoder(textes_off)
synchroniser(); t0 = time.perf_counter()
proba_lstm_off = predire_lstm(modele_lstm, X_off_l)
synchroniser(); temps_inf_lstm_off = time.perf_counter() - t0
evaluer_jeu("BiLSTM", "test", Y_off, proba_lstm_off, None, temps_inf_lstm_off)

BiLSTM inférence: 100%|██████████| 20/20 [00:01<00:00, 15.25it/s]



== BiLSTM [test] ==  inférence 1.3 s (0.13 ms/commentaire) sur 10000 lignes


,seuil,precision,rappel,f1,roc_auc
toxic,0.5,0.2870,0.8665,0.4311,0.8972
severe_toxic,0.5,0.0459,0.9138,0.0875,0.9553
obscene,0.5,0.2420,0.8640,0.3781,0.9266
threat,0.5,0.0185,0.8529,0.0362,0.9054
insult,0.5,0.2113,0.8569,0.3390,0.9136
identity_hate,0.5,0.0620,0.8480,0.1155,0.9004



== BiLSTM [test_opt] ==  inférence 1.3 s (0.13 ms/commentaire) sur 10000 lignes


,seuil,precision,rappel,f1,roc_auc
toxic,0.84,0.4716,0.6014,0.5287,0.8972
severe_toxic,0.96,0.1884,0.4483,0.2653,0.9553
obscene,0.90,0.4838,0.5954,0.5338,0.9266
threat,0.96,0.1014,0.2059,0.1359,0.9054
insult,0.90,0.4447,0.5621,0.4966,0.9136
identity_hate,0.86,0.1168,0.4720,0.1873,0.9004


# Modèle 2 : DistilBERT (jeu d'entraînement commun)

In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

print("Lignes d'entraînement DistilBERT :", len(idx_commun), "(jeu commun)")

tokenizer = AutoTokenizer.from_pretrained(NOM_BERT)


def tokeniser(liste_textes):
    enc = tokenizer(list(liste_textes), truncation=True, padding="max_length",
                    max_length=LONGUEUR_BERT, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]


ids_train_b, masque_train_b = tokeniser(textes[idx_commun])
ids_val_b, masque_val_b = tokeniser(textes[idx_val])
Y_train_b = torch.from_numpy(Y[idx_commun])

Lignes d'entraînement DistilBERT : 30000 (jeu commun)


In [10]:
@torch.no_grad()
def predire_bert(modele, ids, masque, batch=64):
    modele.eval()
    sorties = [torch.sigmoid(modele(input_ids=ids[i:i + batch].to(DEVICE),
                                    attention_mask=masque[i:i + batch].to(DEVICE)).logits).cpu()
               for i in tqdm(range(0, len(ids), batch), desc="DistilBERT inférence")]
    return torch.cat(sorties).numpy()


chemin_bert = DOSSIER_MODELES / "distilbert"
fichier_temps = chemin_bert / "temps_train.json"
if (chemin_bert / "model.safetensors").exists() and not REENTRAINER_BERT:
    # Modèle déjà entraîné sur le même jeu commun (même graine) : on le recharge au lieu de le réentraîner
    modele_bert = AutoModelForSequenceClassification.from_pretrained(chemin_bert).to(DEVICE)
    if fichier_temps.exists():
        temps_train_bert = json.loads(fichier_temps.read_text())["temps_train_s"]
    else:
        temps_train_bert = TEMPS_TRAIN_BERT_CONNU if TEMPS_TRAIN_BERT_CONNU is not None else float("nan")
    print("DistilBERT rechargé depuis", chemin_bert, "| temps d'entraînement :", temps_train_bert)
else:
    fixer_graine()
    modele_bert = AutoModelForSequenceClassification.from_pretrained(
        NOM_BERT, num_labels=len(LABELS), problem_type="multi_label_classification").to(DEVICE)   # float32 par défaut
    perte_b = nn.BCEWithLogitsLoss(pos_weight=calculer_pos_weight(Y[idx_commun]))   # BCE + pos_weight
    optimiseur_b = torch.optim.AdamW(modele_bert.parameters(), lr=LR_BERT, weight_decay=0.01)
    loader_b = DataLoader(TensorDataset(ids_train_b, masque_train_b, Y_train_b), batch_size=BATCH_BERT, shuffle=True)
    n_etapes = EPOCHS_BERT * len(loader_b)
    planificateur = get_linear_schedule_with_warmup(optimiseur_b, int(0.1 * n_etapes), n_etapes)   # 10 % d'échauffement

    synchroniser(); t0 = time.perf_counter()
    for epoque in range(EPOCHS_BERT):
        modele_bert.train()
        total = 0.0
        barre = tqdm(loader_b, desc=f"DistilBERT époque {epoque + 1}/{EPOCHS_BERT}")
        for etape, (ids, masque, yb) in enumerate(barre, 1):
            ids, masque, yb = ids.to(DEVICE), masque.to(DEVICE), yb.to(DEVICE)
            optimiseur_b.zero_grad()
            logits = modele_bert(input_ids=ids, attention_mask=masque).logits
            l = perte_b(logits, yb)
            l.backward()
            optimiseur_b.step()
            planificateur.step()
            total += l.item()
            barre.set_postfix(perte=f"{total / etape:.4f}")   # perte moyenne en direct
        print(f"  perte moyenne : {total / len(loader_b):.4f}")
    synchroniser(); temps_train_bert = time.perf_counter() - t0

    modele_bert.save_pretrained(chemin_bert)
    tokenizer.save_pretrained(chemin_bert)
    fichier_temps.write_text(json.dumps({"temps_train_s": temps_train_bert}))

# Inférence chronométrée sur toute la validation
synchroniser(); t0 = time.perf_counter()
proba_bert = predire_bert(modele_bert, ids_val_b, masque_val_b)
synchroniser(); temps_inf_bert = time.perf_counter() - t0

evaluer_jeu("DistilBERT", "validation", Y[idx_val], proba_bert, temps_train_bert, temps_inf_bert)

W1110 16:25:02.016000 70635 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5080.25it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
DistilBER

  perte moyenne : 0.5420


DistilBERT époque 2/2: 100%|██████████| 1875/1875 [32:33<00:00,  1.04s/it, perte=0.2464]


  perte moyenne : 0.2464


DistilBERT inférence: 100%|██████████| 499/499 [51:55<00:00,  6.24s/it]    



== DistilBERT [validation] ==  inférence 410.1 s (12.85 ms/commentaire) sur 31915 lignes


,seuil,precision,rappel,f1,roc_auc
toxic,0.5,0.5772,0.9280,0.7118,0.9785
severe_toxic,0.5,0.1780,0.9782,0.3012,0.9892
obscene,0.5,0.5226,0.9708,0.6795,0.9899
threat,0.5,0.0459,0.9459,0.0876,0.9864
insult,0.5,0.4355,0.9628,0.5998,0.9843
identity_hate,0.5,0.1076,0.9490,0.1933,0.9815



== DistilBERT [validation_opt] ==  inférence 410.1 s (12.85 ms/commentaire) sur 31915 lignes


,seuil,precision,rappel,f1,roc_auc
toxic,0.92,0.8561,0.7477,0.7983,0.9785
severe_toxic,0.97,0.3655,0.7321,0.4876,0.9892
obscene,0.94,0.8152,0.8461,0.8303,0.9899
threat,0.97,0.4286,0.3649,0.3942,0.9864
insult,0.94,0.7058,0.7788,0.7405,0.9843
identity_hate,0.96,0.4202,0.4660,0.4419,0.9815


In [11]:
# Test officiel commun : DistilBERT
ids_off_b, masque_off_b = tokeniser(textes_off)
synchroniser(); t0 = time.perf_counter()
proba_bert_off = predire_bert(modele_bert, ids_off_b, masque_off_b)
synchroniser(); temps_inf_bert_off = time.perf_counter() - t0
evaluer_jeu("DistilBERT", "test", Y_off, proba_bert_off, None, temps_inf_bert_off)

DistilBERT inférence: 100%|██████████| 157/157 [01:21<00:00,  1.94it/s]


== DistilBERT [test] ==  inférence 81.0 s (8.10 ms/commentaire) sur 10000 lignes


,seuil,precision,rappel,f1,roc_auc
toxic,0.5,0.3649,0.9694,0.5302,0.9619
severe_toxic,0.5,0.0712,0.9828,0.1329,0.9858
obscene,0.5,0.3267,0.9768,0.4896,0.9764
threat,0.5,0.0504,1.0000,0.0959,0.9929
insult,0.5,0.2827,0.9793,0.4388,0.9700
identity_hate,0.5,0.0867,0.9680,0.1591,0.9756



== DistilBERT [test_opt] ==  inférence 81.0 s (8.10 ms/commentaire) sur 10000 lignes


,seuil,precision,rappel,f1,roc_auc
toxic,0.92,0.5737,0.8216,0.6756,0.9619
severe_toxic,0.97,0.2018,0.7586,0.3188,0.9858
obscene,0.94,0.5820,0.7944,0.6718,0.9764
threat,0.97,0.5758,0.5588,0.5672,0.9929
insult,0.94,0.5487,0.7190,0.6224,0.9700
identity_hate,0.96,0.4773,0.5040,0.4903,0.9756


# Modèle 3 : LAYA (Convai Innovations) — zéro-shot

LAYA est un modèle de décision **non autorégressif** (Apache 2.0, backbone ModernBERT-large, 421 M de paramètres) : on lui donne un texte et des questions typées, il répond à toutes en **un seul passage** avec des probabilités. Installation : `pip install laya`.

**Adaptation au multi-label :** une question `noul` (oui/non) par label, soit 6 questions par commentaire ; la probabilité de « oui » sert de score, comme la sortie sigmoïde des autres modèles.

**Pourquoi LAYA n'est pas entraîné.** Le TP demande trois modèles entraînés sur les mêmes données. Pour LAYA, les deux voies d'entraînement ne sont pas réalistes sur un Mac M3 16 Go :
- le **fine-tuning complet** (RLCD) : la documentation annonce 4 à 5 h sur 2 GPU T4 pour 30 000 questions, ici il en faudrait 6 par commentaire, avec un script prévu pour un autre format de données ;
- `agent.fit` (têtes apprises sur le lecteur gelé `vegaml` 0,8 B) : **mesuré à 7,6 s par ligne d'entraînement** et 1,8 s par prédiction, soit environ **63 h** pour 30 000 lignes. Sur 120 lignes, la tête n'était retenue que pour 3 labels sur 6.

LAYA est donc utilisé **tel que publié (zéro-shot)** et évalué sur les mêmes données de test que les autres. La documentation de LAYA indique que ses checkpoints de base sont proches du hasard hors de leur domaine et sur-confiants ; son résultat est un **plancher**, à comparer en gardant à l'esprit qu'il n'a vu aucune donnée d'entraînement.

**Vitesse :** environ 0,2 à 0,4 s par commentaire sur le MPS ; `predict_batch` n'accélère pas (mesuré). Cette cellule dure donc environ 1 h : `N_VAL_LAYA` lignes de validation (pour le choix des seuils) puis tout le test commun.

In [12]:
import laya

# Une question oui/non (type "noul") par label ; l'énoncé de la question décrit le label
QUESTIONS_LAYA = {
    "toxic": "Is this comment toxic?",
    "severe_toxic": "Is this comment severely toxic?",
    "obscene": "Does this comment contain obscene language?",
    "threat": "Does this comment contain a threat?",
    "insult": "Does this comment contain an insult?",
    "identity_hate": "Does this comment contain identity-based hate?",
}
questions = {l: {"type": "noul", "instructions": q} for l, q in QUESTIONS_LAYA.items()}

agent_laya = laya.load("convaiinnovations/laya")   # checkpoint anglais (~808 Mo), téléchargé au premier appel


def predire_laya(liste_textes, cache):
    """Probabilité de « oui » pour chaque label, texte par texte (un seul passage par texte).

    La progression est enregistrée dans `cache` (.npz) toutes les 250 lignes : si le notebook s'arrête, on reprend
    là où on s'était arrêté, et un cache complet évite de refaire le calcul (supprimer le fichier pour le refaire).
    Renvoie (probabilités, temps total en secondes, y compris les reprises).
    """
    n = len(liste_textes)
    sortie = np.zeros((n, len(LABELS)), dtype=np.float32)
    debut, temps_avant = 0, 0.0
    if cache.exists():
        d = np.load(cache)
        if len(d["sortie"]) == n:                       # cache compatible avec ce jeu de textes
            sortie, debut, temps_avant = d["sortie"], int(d["n"]), float(d["temps"])
            print(f"Reprise de {cache.name} : {debut}/{n} lignes déjà calculées")
    t0 = time.perf_counter()
    for i in tqdm(range(debut, n), initial=debut, total=n, desc="LAYA"):
        reponses = agent_laya.predict(liste_textes[i], questions)["answers"]
        sortie[i] = [reponses[l]["noul"] for l in LABELS]
        if (i + 1) % 250 == 0 or i + 1 == n:
            np.savez(cache, sortie=sortie, n=i + 1, temps=temps_avant + time.perf_counter() - t0)
    return sortie, temps_avant + time.perf_counter() - t0


# Pas d'entraînement en zéro-shot : temps d'entraînement = 0.
# Validation : les N_VAL_LAYA premières lignes de la validation (pour choisir les seuils ; la prédiction est lente)
idx_laya = idx_val[:N_VAL_LAYA]
proba_laya, temps_inf_laya = predire_laya(list(textes[idx_laya]), DOSSIER_PRED / "laya_validation_cache.npz")
evaluer_jeu("LAYA (zéro-shot)", "validation", Y[idx_laya], proba_laya, 0.0, temps_inf_laya)

# Test officiel commun : les MÊMES lignes que BiLSTM et DistilBERT
proba_laya_off, temps_inf_laya_off = predire_laya(list(textes_off), DOSSIER_PRED / "laya_test_cache.npz")
evaluer_jeu("LAYA (zéro-shot)", "test", Y_off, proba_laya_off, None, temps_inf_laya_off)


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 489.13it/s]
/Users/situn/Situu/Projects/depl_ai_cyber/.venv/lib/python3.12/site-packages/laya/agent.py:2259: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


Reprise de laya_validation_cache.npz : 3000/3000 lignes déjà calculées


LAYA: 100%|██████████| 3000/3000 [00:00<?, ?it/s]



== LAYA (zéro-shot) [validation] ==  inférence 2191.6 s (730.55 ms/commentaire) sur 3000 lignes


,seuil,precision,rappel,f1,roc_auc
toxic,0.5,0.8524,0.7778,0.8134,0.9768
severe_toxic,0.5,0.1143,1.0000,0.2051,0.9882
obscene,0.5,0.6324,0.7222,0.6744,0.9723
threat,0.5,0.0195,0.8333,0.0380,0.8975
insult,0.5,0.4422,0.9745,0.6083,0.9834
identity_hate,0.5,0.1543,0.9355,0.2648,0.9691



== LAYA (zéro-shot) [validation_opt] ==  inférence 2191.6 s (730.55 ms/commentaire) sur 3000 lignes


... (3 outputs left out)


== LAYA (zéro-shot) [test] ==  inférence 7006.1 s (700.61 ms/commentaire) sur 10000 lignes


,seuil,precision,rappel,f1,roc_auc
toxic,0.5,0.5390,0.8593,0.6625,0.9611
severe_toxic,0.5,0.0348,0.9828,0.0673,0.9814
obscene,0.5,0.3853,0.6434,0.4820,0.9310
threat,0.5,0.0348,1.0000,0.0673,0.9950
insult,0.5,0.2909,0.9690,0.4475,0.9661
identity_hate,0.5,0.1401,0.9280,0.2434,0.9767



== LAYA (zéro-shot) [test_opt] ==  inférence 7006.1 s (700.61 ms/commentaire) sur 10000 lignes


,seuil,precision,rappel,f1,roc_auc
toxic,0.50,0.5390,0.8593,0.6625,0.9611
severe_toxic,0.76,0.1723,0.7069,0.2770,0.9814
obscene,0.52,0.3966,0.6269,0.4859,0.9310
threat,0.90,0.2417,0.8529,0.3766,0.9950
insult,0.69,0.4701,0.8276,0.5996,0.9661
identity_hate,0.80,0.3757,0.5680,0.4522,0.9767


# Tableau comparatif final

In [13]:
# Tableau résumé : moyennes (macro) sur les 6 labels
LIGNES_TRAIN = {"BiLSTM": len(idx_commun), "DistilBERT": len(idx_commun), "LAYA (zéro-shot)": 0}
lignes = []
for nom, res in RESULTATS.items():
    ligne = {"modèle": nom, "lignes d'entraînement": LIGNES_TRAIN.get(nom, 0), "entraînement (s)": res["temps_train_s"]}
    if "validation" in res:
        ligne["ROC-AUC (val)"] = res["validation"]["par_label"]["roc_auc"].mean()
        ligne["F1 (val, seuil optimisé)"] = res["validation_opt"]["par_label"]["f1"].mean()
        ligne["inférence val (s)"] = res["validation"]["temps_inference_s"]
        ligne["lignes (val)"] = res["validation"]["n_lignes"]
    if "test" in res:
        ligne["ROC-AUC (test)"] = res["test"]["par_label"]["roc_auc"].mean()
        for suffixe, jeu in (("seuil 0,5", "test"), ("seuil optimisé", "test_opt")):
            moy = res[jeu]["par_label"].mean()
            ligne[f"précision (test, {suffixe})"] = moy["precision"]
            ligne[f"rappel (test, {suffixe})"] = moy["rappel"]
            ligne[f"F1 (test, {suffixe})"] = moy["f1"]
        ligne["inférence test (s)"] = res["test"]["temps_inference_s"]
        ligne["ms/commentaire (test)"] = res["test"]["ms_par_commentaire"]
        ligne["lignes (test)"] = res["test"]["n_lignes"]
    lignes.append(ligne)
resume = pd.DataFrame(lignes).set_index("modèle")
display(resume.round(4))

# Détail par label (validation et test, seuil 0,5 et seuil optimisé)
detail = pd.concat({(nom, jeu): res[jeu]["par_label"] for nom, res in RESULTATS.items()
                    for jeu in ("validation", "validation_opt", "test", "test_opt") if jeu in res},
                   names=["modèle", "jeu", "label"])
display(detail.round(4))

# Sauvegarde des résultats + écriture du tableau dans CLAUDE.md (entre les marqueurs)
resume.to_csv(DOSSIER_MODELES / "resultats.csv")
detail.to_csv(DOSSIER_MODELES / "resultats_par_label.csv")
# Seuils par label, choisis sur la validation : à réutiliser dans l'API (score 0-100)
chemin_seuils = DOSSIER_MODELES / "seuils.json"
seuils_fichier = json.loads(chemin_seuils.read_text(encoding="utf-8")) if chemin_seuils.exists() else {}
seuils_fichier.update({nom: dict(zip(LABELS, map(float, res["seuils"]))) for nom, res in RESULTATS.items() if "seuils" in res})
chemin_seuils.write_text(json.dumps(seuils_fichier, indent=1), encoding="utf-8")   # fusion : ne perd pas les seuils des autres modèles


def tableau_markdown(d: pd.DataFrame) -> str:
    entetes = [d.index.name] + list(d.columns)
    lignes_md = ["| " + " | ".join(entetes) + " |", "|" + "---|" * len(entetes)]
    for idx, ligne in d.iterrows():
        lignes_md.append("| " + " | ".join([str(idx)] + [f"{v:.4g}" for v in ligne]) + " |")
    return "\n".join(lignes_md)


DEBUT, FIN = "<!-- RESULTATS:DEBUT -->", "<!-- RESULTATS:FIN -->"
contenu = CHEMIN_CLAUDE_MD.read_text(encoding="utf-8")
avant, reste = contenu.split(DEBUT)
_, apres = reste.split(FIN)
nouveau = (f"{avant}{DEBUT}\n\nMacro-moyenne sur les 6 labels. Entraînement : mêmes {len(idx_commun)} lignes (LAYA : zéro-shot). "
           f"**val** = 20 % de train ; **test** = {len(textes_off)} lignes du test officiel Kaggle (sans les -1), les mêmes pour les trois modèles. "
           f"Seuil optimisé = seuil par label maximisant le F1 sur la validation.\n\n"
           f"{tableau_markdown(resume)}\n\n{FIN}{apres}")
CHEMIN_CLAUDE_MD.write_text(nouveau, encoding="utf-8")
print("Tableau écrit dans", CHEMIN_CLAUDE_MD)

,lignes d'entraînement,entraînement (s),ROC-AUC (val),"F1 (val, seuil optimisé)",inférence val (s),lignes (val),ROC-AUC (test),"précision (test, seuil 0,5)","rappel (test, seuil 0,5)","F1 (test, seuil 0,5)","précision (test, seuil optimisé)","rappel (test, seuil optimisé)","F1 (test, seuil optimisé)",inférence test (s),ms/commentaire (test),lignes (test)
modèle,,,,,,,,,,,,,,,,
BiLSTM,30000,157.3706,0.9478,0.4431,4.1781,31915,0.9164,0.1444,0.8670,0.2312,0.3011,0.4808,0.3579,1.3143,0.1314,10000
DistilBERT,30000,3859.8226,0.9850,0.6155,410.0706,31915,0.9771,0.1971,0.9794,0.3078,0.4932,0.6927,0.5577,81.0291,8.1029,10000
LAYA (zéro-shot),0,0.0000,0.9646,0.5767,2191.6424,3000,0.9686,0.2375,0.8971,0.3283,0.3659,0.7403,0.4756,7006.1384,700.6138,10000


seuil  precision  rappel      f1  \
modèle           jeu        label                                             
BiLSTM           validation toxic           0.50     0.3547  0.8658  0.5032   
                            severe_toxic    0.50     0.1207  0.9221  0.2135   
                            obscene         0.50     0.3135  0.9055  0.4658   
                            threat          0.50     0.0182  0.8784  0.0357   
                            insult          0.50     0.2649  0.8984  0.4092   
...                                          ...        ...     ...     ...   
LAYA (zéro-shot) test_opt   severe_toxic    0.76     0.1723  0.7069  0.2770   
                            obscene         0.52     0.3966  0.6269  0.4859   
                            threat          0.90     0.2417  0.8529  0.3766   
                            insult          0.69     0.4701  0.8276  0.5996   
                            identity_hate   0.80     0.3757  0.5680  0.4522   

                                           roc_auc  
modèle           jeu        label                   
BiLSTM           validation toxic           0.9297  
                            severe_toxic    0.9764  
                            obscene         0.9576  
                            threat          0.9470  
                            insult          0.9502  
...                                            ...  
LAYA (zéro-shot) test_opt   severe_toxic    0.9814  
                            obscene         0.9310  
                            threat          0.9950  
                            insult          0.9661  
                            identity_hate   0.9767  

[72 rows x 5 columns]

Tableau écrit dans /Users/situn/Situu/Projects/depl_ai_cyber/CLAUDE.md
